# Stage 5 · Reasoning & Test-Time Compute — EXERCISES
### Topics: Best-of-N · RLVR · DAPO Fixes · Entropy Collapse · Length Hacking · Training Diagnostics

> Fill every `# TODO`. Run `# ASSERT` cells to verify.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import re
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass, field


---
## 1 · Best-of-N Sampling & Test-Time Compute Scaling

Instead of (or in addition to) better training, we can spend more compute at **inference time**.

### Best-of-N (BoN / rejection sampling)
1. Sample N independent completions from the policy
2. Score each with a reward model or verifiable function
3. Return the highest-scoring one

**Key scaling result (Snell et al., 2024):**
Performance scales as **O(log N)** — diminishing returns but consistent gains.

### Weighted Best-of-N (WBoN)
Instead of hard argmax, weight samples by exponentiated reward:
$$w_i \propto \pi_{ref}(y_i|x) \cdot \exp(r(x,y_i)/\beta)$$

This is the **optimal RLHF policy** — WBoN at inference *approximates* what RLHF training achieves.

### BoN vs Beam Search
| | BoN | Beam Search |
|---|---|---|
| Diversity | High (independent) | Low (pruned tree) |
| Parallelisable | Yes | Partially |
| RM dependent | Yes | Optional |
| Optimal | No | Locally |


In [ ]:
def best_of_n(
    completions: List[str],
    rewards:     torch.Tensor,   # (N,)
) -> Tuple[str, int, float]:
    """Return (best_completion, best_index, best_reward). Use rewards.argmax()."""
    # TODO
    raise NotImplementedError


def weighted_best_of_n_weights(
    log_probs_ref: torch.Tensor,  # (N,)
    rewards:       torch.Tensor,  # (N,)
    beta:          float = 1.0,
) -> torch.Tensor:                # (N,) — sums to 1
    """
    log_weights = log_probs_ref + rewards / beta
    return softmax(log_weights, dim=0)
    """
    # TODO
    raise NotImplementedError


def simulate_bon_scaling(
    reward_fn,
    n_values:  List[int],
    n_trials:  int = 500,
    seed:      int = 0,
) -> Dict[int, float]:
    """
    For each N: run n_trials experiments, draw N rewards, take max.
    Return {N: mean_max_reward}.
    """
    # TODO
    raise NotImplementedError


def bon_marginal_efficiency(bon_rewards: Dict[int, float]) -> Dict[int, float]:
    """efficiency(N) = (r_N - r_1) / (N - 1)"""
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(42)
N = 10
completions = [f"c_{i}" for i in range(N)]
rewards     = torch.randn(N)
best_c, best_i, best_r = best_of_n(completions, rewards)
assert best_r == rewards.max().item()

w = weighted_best_of_n_weights(torch.randn(N), rewards, beta=1.0)
assert abs(w.sum().item() - 1.0) < 1e-5 and (w >= 0).all()

w_lo = weighted_best_of_n_weights(torch.randn(N), rewards, beta=0.01)
w_hi = weighted_best_of_n_weights(torch.randn(N), rewards, beta=100.0)
assert w_lo.max() > w_hi.max(), "Low beta should concentrate weights"

scaling = simulate_bon_scaling(lambda: float(np.random.randn()), [1, 4, 16, 64], n_trials=300)
assert scaling[64] > scaling[4] > scaling[1]
print(f"best_of_n ✓  weighted_best_of_n ✓  simulate_bon_scaling ✓")
print(f"  N=1:{scaling[1]:.3f}  N=4:{scaling[4]:.3f}  N=16:{scaling[16]:.3f}  N=64:{scaling[64]:.3f}")


---
## 2 · RLVR — RL with Verifiable Rewards

**RLVR** is the training paradigm behind DeepSeek-R1 and similar reasoning models.  
Replace the learned reward model with a **deterministic verifier** — the reward cannot be hacked.

### The RLVR loop
```
for each batch of (prompt, ground_truth) pairs:
    1. Sample G completions per prompt (e.g., G=8)
    2. Verify each: reward = verifier(completion, ground_truth)  ← 0 or 1
    3. Compute group-relative advantages (GRPO)
    4. Update policy with clipped surrogate + KL penalty
```

### Cold-start problem
At the start of training, the model rarely gets correct answers → all rewards = 0  
→ all advantages = 0 → **no gradient signal**.

**Solutions:**
1. **SFT warm-up:** supervised fine-tune on worked examples before RL
2. **Curriculum:** start with easy problems, increase difficulty
3. **Rejection sampling fine-tuning (RFT):** generate many candidates, keep correct ones, SFT on them
4. **Format reward:** give partial credit just for using `<think>...</think>` tags

### The DeepSeek-R1 emergent CoT finding
With *only* verifiable rewards and no supervised CoT data, models spontaneously develop:
- Self-correction: backtracking and revising
- Exploration: trying multiple approaches
- Reflection: "Wait, let me reconsider..."


In [ ]:
def rlvr_reward(
    completion:   str,
    ground_truth: str,
    format_bonus: float = 0.1,
) -> float:
    """
    1. Extract final number from both strings using re patterns:
       try '#### N', '= N', then last number in string
    2. Return 0.0 if either extraction fails
    3. Return 1.0 if |pred - truth| < 1e-5
    4. Add format_bonus if <think>...</think> present in order
    """
    def extract_answer(text: str) -> Optional[float]:
        # TODO: try three regex patterns, return float or None
        raise NotImplementedError
    # TODO: extract, compare, add bonus
    raise NotImplementedError


def grpo_advantages_for_rlvr(
    rewards: torch.Tensor,   # (B*G,)
    G:       int,
    eps:     float = 1e-8,
) -> torch.Tensor:           # (B*G,)
    """
    Group-relative normalisation.
    Key: if std ≈ 0 (all rewards identical), return zeros — don't divide by ~0.
    Hint: mask = (std > eps).float(); adv = mask * (rg - mean) / (std + eps)
    """
    # TODO
    raise NotImplementedError


def rejection_sampling_filter(
    completions:   List[str],
    ground_truths: List[str],
    threshold:     float = 1.0,
) -> List[Tuple[str, str]]:
    """Keep (gt, completion) pairs where rlvr_reward >= threshold."""
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
cot_correct  = "<think>2+2=4</think> #### 4"
bare_correct = "#### 4"
assert rlvr_reward(cot_correct,  "#### 4") == 1.1, f"Got {rlvr_reward(cot_correct, '#### 4')}"
assert rlvr_reward(bare_correct, "#### 4") == 1.0
assert rlvr_reward("#### 5",     "#### 4") == 0.0

B, G = 3, 4
adv_same = grpo_advantages_for_rlvr(torch.zeros(B*G), G)
assert adv_same.abs().max() < 1e-5, "All-same rewards → zero advantages"

adv_varied = grpo_advantages_for_rlvr(torch.randn(B*G), G)
for i in range(B):
    assert adv_varied[i*G:(i+1)*G].mean().abs() < 1e-4

comps  = ["<think>ok</think> #### 4", "#### 5", "<think>ok</think> #### 4"]
truths = ["#### 4", "#### 4", "#### 4"]
kept   = rejection_sampling_filter(comps, truths, threshold=1.0)
assert len(kept) == 2
print("rlvr_reward ✓  grpo_advantages_for_rlvr ✓  rejection_sampling_filter ✓")


---
## 3 · DAPO — Four Fixes for GRPO at Scale (ByteDance, 2025)

DAPO identifies and resolves four failure modes that appear when scaling GRPO to hard reasoning tasks.

### Fix 1: Token-level loss normalisation
GRPO averages loss over sequences → long sequences contribute less per-token.  
**DAPO:** normalise loss by **total completion tokens** across the batch, not by sequences.

### Fix 2: Clip-higher (asymmetric clipping)
Standard PPO clips both increases and decreases equally.  
For sparse verifiable rewards, clipping ratio *increases* on positive advantages is too conservative.  
**DAPO:** use $\varepsilon_{high} > \varepsilon_{low}$:
$$\text{clip}(\rho, 1 - \varepsilon_{low},\ 1 + \varepsilon_{high\ \text{if}\  A>0\ \text{else}\ low})$$

### Fix 3: Entropy bonus to prevent collapse
Add $-\alpha \cdot H(\pi)$ directly to the loss.  
Since $H$ is differentiable, this creates a gradient that pushes the policy toward higher entropy.

### Fix 4: Overlong sequence filtering
Models learn to generate very long responses (reward hacking via length).  
**DAPO:** filter sequences exceeding `max_length`; replace their rewards with a penalty.


In [ ]:
def token_level_policy_loss(
    logits:     torch.Tensor,   # (B, T, V)
    input_ids:  torch.Tensor,   # (B, T)
    advantages: torch.Tensor,   # (B,)  detached
    comp_mask:  torch.Tensor,   # (B, T)
    logits_old: torch.Tensor,   # (B, T, V)  detached
    epsilon:    float = 0.2,
) -> torch.Tensor:
    """
    DAPO Fix 1: token-level loss normalised by TOTAL completion tokens.
    Steps:
      1. log_softmax both logits → gather per-token log-probs → (B,T) each
      2. ratio = exp(lp_new - lp_old)
      3. broadcast advantages: adv_tok = advantages.unsqueeze(-1).expand_as(ratio)
      4. -min(ratio*adv, clamp(ratio, 1-ε, 1+ε)*adv)    → (B,T)
      5. apply comp_mask; divide by total comp tokens (not B)
    """
    # TODO
    raise NotImplementedError


def dapo_clip_loss(
    log_probs_new: torch.Tensor,  # (B,)
    log_probs_old: torch.Tensor,  # (B,)  detached
    advantages:    torch.Tensor,  # (B,)  detached
    eps_low:  float = 0.2,
    eps_high: float = 0.28,
) -> torch.Tensor:
    """
    DAPO Fix 2: asymmetric clip.
    hi = 1 + eps_high where advantages > 0, else 1 + eps_low.
    Use torch.where to construct hi tensor.
    """
    # TODO
    raise NotImplementedError


def entropy_from_logits(
    logits:    torch.Tensor,   # (B, T, V)
    comp_mask: torch.Tensor,   # (B, T)
) -> torch.Tensor:             # scalar
    """
    DAPO Fix 3: entropy term for the bonus.
    H_t = -(p * log_p).sum(-1); mean over completion tokens.
    """
    # TODO
    raise NotImplementedError


def filter_overlong(
    rewards:    torch.Tensor,   # (B,)
    lengths:    torch.Tensor,   # (B,)
    max_length: int   = 4096,
    penalty:    float = -1.0,
) -> torch.Tensor:
    """DAPO Fix 4: torch.where(lengths > max_length, penalty, rewards)."""
    # TODO
    raise NotImplementedError


def dapo_total_loss(
    logits:            torch.Tensor,  # (B, T, V)
    input_ids:         torch.Tensor,  # (B, T)
    logits_old:        torch.Tensor,  # (B, T, V)  detached
    log_probs_seq:     torch.Tensor,  # (B,)
    log_probs_old_seq: torch.Tensor,  # (B,)  detached
    advantages:        torch.Tensor,  # (B,)  detached
    comp_mask:         torch.Tensor,  # (B, T)
    eps_low:   float = 0.2,
    eps_high:  float = 0.28,
    ent_coef:  float = 0.01,
) -> Tuple[torch.Tensor, Dict]:
    """L = token_level_policy_loss - ent_coef * entropy. Return (total, info dict)."""
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B, T, V = 4, 12, 50
logits     = torch.randn(B, T, V, requires_grad=True)
input_ids  = torch.randint(0, V, (B, T))
comp_mask  = torch.cat([torch.zeros(B,4), torch.ones(B,8)], dim=1).long()
adv        = torch.randn(B)
logits_old = logits.detach().clone()
lp_new     = torch.randn(B, requires_grad=True)
lp_old     = lp_new.detach()

tok_loss  = token_level_policy_loss(logits, input_ids, adv, comp_mask, logits_old)
assert tok_loss.shape == ()

loss_dapo = dapo_clip_loss(lp_new, lp_old, adv)
assert loss_dapo.shape == ()

ent = entropy_from_logits(logits, comp_mask)
assert 0 < ent.item() <= math.log(V) + 1e-4

filtered = filter_overlong(torch.ones(4), torch.tensor([100,200,5000,300]), max_length=4096)
assert filtered[2].item() == -1.0 and filtered[0].item() == 1.0

total, info = dapo_total_loss(logits, input_ids, logits_old, lp_new, lp_old, adv, comp_mask)
total.backward()
assert logits.grad is not None, "Gradient must flow through logits"
print("token_level_policy_loss ✓  dapo_clip_loss ✓  entropy_from_logits ✓")
print("filter_overlong ✓  dapo_total_loss ✓")


---
## 4 · Simulating Entropy Collapse

Entropy collapse is the most common silent failure in RLVR training at scale.

### What happens step by step
1. Policy gradient pushes mass toward high-reward tokens
2. High-probability tokens dominate; entropy drops
3. The model explores less → discovers fewer correct solutions
4. Reward signal becomes even sparser → gradient nearly zero
5. Model is stuck — can't improve

### The entropy-exploitation tradeoff
| Entropy | Exploration | Exploitation |
|---|---|---|
| High (uniform) | Maximum | None |
| Medium | Balanced | Balanced |
| Low (peaked) | None | Maximum |
| Collapsed (→0) | None | Wrong solution |

### Monitoring thresholds (practical)
- **Warning:** entropy < 0.5 nats (for vocab size V=50k, max entropy ≈ 10.8 nats)
- **Critical:** entropy < 0.1 nats — essentially deterministic
- **Healthy range:** 0.5×H_max to 0.9×H_max


In [ ]:
def simulate_entropy_dynamics(
    n_steps:     int   = 120,
    ent_coef:    float = 0.0,
    reward_temp: float = 1.0,
    V:           int   = 30,
    seed:        int   = 42,
) -> Dict[str, List[float]]:
    """
    Toy entropy simulation.
    - logits: (V,) starts at zeros
    - Each step: probs = softmax(logits); entropy = -sum(p * log_p)
    - rl_loss = -reward_temp * log_probs[0]   (push prob toward token 0)
    - loss    = rl_loss - ent_coef * entropy
    - SGD step; track entropy and p_best (probs[0]) histories
    """
    # TODO
    raise NotImplementedError


def detect_entropy_collapse(
    entropy_history: List[float],
    window:    int   = 10,
    threshold: float = 0.3,
) -> Tuple[bool, int]:
    """
    Scan rolling windows of size `window` over entropy_history.
    If any window has mean < threshold, return (True, start_of_window_step).
    Else return (False, -1).
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
coefs = [0.0, 0.02, 0.1]
sims  = {c: simulate_entropy_dynamics(n_steps=100, ent_coef=c) for c in coefs}

assert sims[0.0]["entropy"][-1] < 0.2,      "No bonus → entropy collapses"
assert sims[0.1]["entropy"][-1] > sims[0.0]["entropy"][-1], "Bonus resists collapse"

collapsed, step = detect_entropy_collapse(sims[0.0]["entropy"], window=5, threshold=0.2)
assert collapsed, "Should detect collapse in the no-bonus run"

print("simulate_entropy_dynamics ✓  detect_entropy_collapse ✓")
for c, hist in sims.items():
    print(f"  ent_coef={c:.2f}: H {hist['entropy'][0]:.3f} → {hist['entropy'][-1]:.3f}")


---
## 5 · End-to-End Training Monitoring Dashboard

Everything you need to monitor a RLVR/GRPO training run at scale.

### Minimum monitoring checklist

| Metric | Healthy range | Action if unhealthy |
|---|---|---|
| `reward_mean` | Increasing steadily | — |
| `reward_nonzero_frac` | > 5% | Curriculum; SFT warm-up |
| `kl_from_ref` | < 5 nats | Increase β; reduce lr |
| `entropy` | > 0.5 | Increase `ent_coef` |
| `clip_fraction` | 0.1–0.3 | Reduce lr if > 0.5 |
| `advantage_std` | ≈ 1.0 | Check normalisation |
| `length_mean` | Stable | Add overlong filter if increasing |

### The KL–Reward Pareto frontier
A well-trained model maximises reward while minimising KL.  
Plot `kl_from_ref` vs `reward_mean` to see if KL is "buying" proportional reward gains.

### Length hacking signature
```
reward_mean ↑ (proxy reward increasing)
length_mean ↑ (completions getting longer)
reward_nonzero_frac stable (same fraction correct, just longer)
```
This pattern → model is hacking length, not learning to reason better.


In [ ]:
@dataclass
class StepMetrics:
    step: int
    reward_mean: float;          reward_std: float
    reward_nonzero_frac: float;  kl_from_ref: float
    entropy: float;              clip_fraction: float
    advantage_std: float;        length_mean: float
    length_std: float


def compute_step_metrics(
    rewards:       torch.Tensor,   # (B,)
    log_probs_new: torch.Tensor,   # (B,)
    log_probs_old: torch.Tensor,   # (B,)
    log_probs_ref: torch.Tensor,   # (B,)
    advantages:    torch.Tensor,   # (B,)
    lengths:       torch.Tensor,   # (B,)
    entropy:       float,
    step:          int,
    epsilon:       float = 0.2,
) -> StepMetrics:
    """
    Compute all monitoring metrics for one training step.
    ratio     = exp(lp_new - lp_old)
    clip_frac = fraction where |ratio - 1| > epsilon
    kl        = mean(lp_new - lp_ref)
    """
    # TODO
    raise NotImplementedError


def health_check(m: StepMetrics) -> List[str]:
    """
    Return warning strings for:
      entropy < 0.5,  kl > 5.0,  reward_nonzero_frac < 0.05,
      clip_fraction > 0.5,  advantage_std outside (0.5, 3.0)
    """
    # TODO
    raise NotImplementedError


def detect_length_hacking(history: List[StepMetrics], window: int = 10) -> bool:
    """
    True if over the last `window` steps vs the `window` steps before:
      reward_mean increased AND length_mean increased >20%
      AND reward_nonzero_frac changed <5pp (not getting more correct, just longer)
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 16
history: List[StepMetrics] = []

for step in range(15):
    m = compute_step_metrics(
        rewards=torch.rand(B)*0.5+step*0.01, log_probs_new=torch.randn(B)*0.1,
        log_probs_old=torch.randn(B)*0.1, log_probs_ref=torch.randn(B)*0.1,
        advantages=torch.randn(B), lengths=torch.randint(100,300,(B,)),
        entropy=1.5-step*0.03, step=step)
    history.append(m)

for step in range(15, 25):
    m = compute_step_metrics(
        rewards=torch.rand(B)*0.5+0.15+(step-15)*0.02, log_probs_new=torch.randn(B)*0.1,
        log_probs_old=torch.randn(B)*0.1, log_probs_ref=torch.randn(B)*0.1,
        advantages=torch.randn(B),
        lengths=torch.randint(200+(step-15)*100, 400+(step-15)*100, (B,)),
        entropy=1.0, step=step)
    history.append(m)

assert detect_length_hacking(history, window=5), "Should detect length hacking"
print("compute_step_metrics ✓  health_check ✓  detect_length_hacking ✓")
print(f"  Final metrics: reward={history[-1].reward_mean:.3f}  length={history[-1].length_mean:.0f}")
